# Fase 1 · Comprensión del problema

**Equipo 1 · Ventas de retail** — Análisis de Datos, ITM 2026-2

**Fuente:** [Superstore Sales Dataset — Kaggle (rohitsahoo/sales-forecasting)](https://www.kaggle.com/datasets/rohitsahoo/sales-forecasting), archivo `train.csv`.

## 1. Contexto del dataset

`train.csv` es la variante **solo-ingresos** del dataset público *Superstore*, un
minorista estadounidense de suministros de oficina, mobiliario y tecnología que
vende a tres segmentos de comprador. Es el caso de estudio estándar de ventas de
*retail* porque combina las tres dimensiones que definen el desempeño comercial
de una operación minorista: **qué se vende** (categoría y subcategoría), **a quién**
(segmento y cliente) y **dónde y cómo** (región, estado, ciudad y modo de envío).

| Dimensión | Contenido |
|---|---|
| Grano de la tabla | Una línea de pedido (un producto dentro de un pedido) |
| Medida disponible | `Sales` (ingreso de la línea, en USD) |
| Identificadores | `Order ID`, `Customer ID`, `Product ID`, `Row ID` |
| Jerarquías | Categoría → Subcategoría → Producto; Región → Estado → Ciudad |
| Tiempo | `Order Date` y `Ship Date` (formato `dd/mm/aaaa`), de enero de 2015 a diciembre de 2018 |
| Clasificadores | `Segment`, `Ship Mode` |

**Advertencia metodológica de primer orden.** Esta variante del dataset **no incluye
`Profit`, `Quantity` ni `Discount`**: el encabezado termina en la columna `Sales`
(se verifica en la celda de código siguiente).

Esto condiciona todo el proyecto: **no es posible calcular margen, utilidad ni
efecto del descuento**. El análisis se orienta, por tanto, a los determinantes del
**ingreso** y del **valor de cliente**, y sustituye las métricas de rentabilidad
por métricas de concentración, frecuencia y estacionalidad. Cualquier conclusión
sobre rentabilidad queda explícitamente fuera del alcance y se declara en la
sección de limitaciones.

In [1]:
from pathlib import Path

import pandas as pd

ruta = Path("../datos/train.csv")
if not ruta.exists():  # en Google Colab, train.csv se sube a la carpeta de la sesión
    ruta = Path("train.csv")

df = pd.read_csv(ruta)

print(f"Filas: {df.shape[0]:,} · Columnas: {df.shape[1]}")
print("Columnas:", list(df.columns))

fechas = pd.to_datetime(df["Order Date"], format="%d/%m/%Y")
print(f"Rango de Order Date: {fechas.min():%Y-%m-%d} a {fechas.max():%Y-%m-%d}")

print(f"Row ID único (grano = línea de pedido): {df['Row ID'].is_unique}")
print(f"Pedidos: {df['Order ID'].nunique():,} · Clientes: {df['Customer ID'].nunique():,} · Productos: {df['Product ID'].nunique():,}")
print("Columnas de rentabilidad presentes:", [c for c in ["Profit", "Quantity", "Discount"] if c in df.columns])

df.isna().sum().loc[lambda s: s > 0]

Filas: 9,800 · Columnas: 18
Columnas: ['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales']
Rango de Order Date: 2015-01-03 a 2018-12-30
Row ID único (grano = línea de pedido): True
Pedidos: 4,922 · Clientes: 793 · Productos: 1,861
Columnas de rentabilidad presentes: []


Postal Code    11
dtype: int64

**Observaciones de calidad identificadas** (se tratan en la Fase 3, no aquí):

- `Postal Code` tiene valores nulos y se lee como numérico decimal; debe tratarse como texto.
- Existe al menos una fila duplicada si se ignora `Row ID`.
- Algunos `Product ID` están asociados a más de un `Product Name`, lo que afecta los análisis a nivel de producto.
- `Country` tiene un único valor (`United States`), por lo que no aporta variación analítica.

## 2. Problema de análisis y de negocio

**Problema de análisis.** Determinar qué factores —volumen de pedidos, valor del
pedido, mezcla de producto, territorio y momento del año— explican la evolución
de los ingresos entre 2015 y 2018, y cuantificar cuáles son sostenibles.

**Problema de negocio.** La dirección observa que los ingresos crecen con fuerza en
los dos últimos años, pero no sabe si ese crecimiento proviene de una mejora del
valor por cliente (palanca sana y sostenible) o de un aumento del número de
transacciones con ticket estancado (palanca que se agota si el padrón de clientes
deja de crecer). Tampoco tiene claridad sobre en qué territorios y categorías
concentrar el esfuerzo comercial del próximo año, ni sobre cómo dimensionar la
operación para el pico de fin de año.

**Decisión que habilita.** Priorizar la asignación de esfuerzo comercial y
logístico para 2019 con evidencia estadística: dónde invertir en adquisición,
dónde defender la base instalada, qué categorías sostienen la facturación y cómo
dimensionar el cuarto trimestre.

## 3. Preguntas clave de análisis

| # | Pregunta | KPI que la responde | Fase donde se resuelve |
|---|---|---|---|
| P1 | ¿El crecimiento proviene del volumen de pedidos o del valor del pedido? | Variación de pedidos vs. variación de ticket medio | 4 y 6 |
| P2 | ¿Qué regiones concentran los ingresos y cuáles crecen más rápido? | Cuota de ingresos por región y crecimiento 2015-2018 | 4 y 6 |
| P3 | ¿Está la facturación concentrada en pocas categorías, productos y clientes? | Cuota del top-3, índice de concentración (HHI) y Gini | 4 y 6 |
| P4 | ¿Existe un patrón estacional y qué dimensión debe tener la operación en el pico? | Índice de estacionalidad mensual y factor Q4 | 4 y 6 |
| P5 | ¿Los pedidos con modo de envío premium (*First Class* / *Same Day*) son de mayor valor? | Chi-cuadrado modo de envío × tramo de valor del pedido | 6 |